# TT-Shape Selection Demo

This notebook shows how to call the local `ttshape_selection` package with multiple target weights.

In [ ]:
from pathlib import Path
import sys

phase_root = Path.cwd()
if not (phase_root / 'ttshape_selection').exists():
    phase_root = Path('/home/pkunwar/characterize_ttlora/phases/ttshape_selection')
sys.path.insert(0, str(phase_root))

phase_root

In [ ]:
from ttshape_selection import WeightSpec, generate_catalog

weights = [
    WeightSpec(name='c_attn', shape=(2304, 768)),
    WeightSpec(name='c_proj', shape=(768, 768)),
]

catalog = generate_catalog(
    weights,
    rank=6,
    core_counts=[2, 3, 4],
    split_strategy='all',
    top_k_per_weight=8,
)
catalog

In [ ]:
for weight_name, candidates in catalog.candidates_by_weight.items():
    print('\n', weight_name)
    for candidate in candidates[:5]:
        print({
            'core_count': candidate.core_count,
            'input_factors': candidate.input_factors,
            'output_factors': candidate.output_factors,
            'tt_shape': candidate.tt_shape,
            'parameter_count': candidate.parameter_count,
            'score': round(candidate.score, 4),
            'balance_penalty': round(candidate.balance_penalty, 4),
            'core_size_penalty': round(candidate.core_size_penalty, 4),
            'compression_ratio': round(candidate.compression_ratio, 2),
        })

In [ ]:
output_path = phase_root / 'examples' / 'demo_candidates.json'
import json
output_path.write_text(json.dumps(catalog.to_dict(), indent=2), encoding='utf-8')
output_path